In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sqlite3
import pandas as pd
import re
from datetime import datetime

Mounted at /content/drive


In [2]:
phase4_path = "/content/drive/MyDrive/Capstone Project/partnerlens_synthetic_data_package/partnerlens_data_package/outputs/phase4_sql_baseline"

phase5_path = "/content/drive/MyDrive/Capstone Project/partnerlens_synthetic_data_package/partnerlens_data_package/outputs/phase5_assistant_prototype"

os.makedirs(phase5_path, exist_ok=True)

db_path = os.path.join(phase4_path, "partnerlens.db")

print("Database path:", db_path)
print("Phase 5 output path:", phase5_path)

Database path: /content/drive/MyDrive/Capstone Project/partnerlens_synthetic_data_package/partnerlens_data_package/outputs/phase4_sql_baseline/partnerlens.db
Phase 5 output path: /content/drive/MyDrive/Capstone Project/partnerlens_synthetic_data_package/partnerlens_data_package/outputs/phase5_assistant_prototype


In [3]:
conn = sqlite3.connect(db_path)

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

tables

,name
0,partner_master
1,partner_pricing
2,partner_metrics


In [4]:
def run_sql(query):
    """
    Runs a SQL query against the PartnerLens SQLite database.
    Returns the result as a pandas DataFrame.
    """
    return pd.read_sql_query(query, conn)

In [5]:
run_sql("SELECT * FROM partner_master LIMIT 5;")

,partner_id,partner_name,partner_type,industry_vertical,partner_size,partner_status,country,state,city,region,integration_type,sales_channel,risk_tier,kyc_status,pci_level,onboarding_date,relationship_manager_region,synthetic_record_flag,industry,status
0,P000001,NorthStar Commerce 0001,Technology Platform,Travel,SMB,Suspended,US,NY,New York,Northeast,API,Direct,High,Approved,Level 3,2019-06-15,Northeast,1,Travel,Suspended
1,P000002,Ironwood Market 0002,ISV,SaaS,Mid-Market,Pilot,US,VA,Norfolk,South,SDK,Direct,Low,Approved,Level 4,2020-02-13,South,1,SaaS,Pilot
2,P000003,Evergreen Commerce 0003,Technology Platform,Professional Services,SMB,Active,US,CO,Colorado Springs,West,API,Direct,Low,Approved,Level 3,2023-03-12,West,1,Professional Services,Active
3,P000004,Summit Platform 0004,ISO/Reseller,Gaming,SMB,Active,US,GA,Augusta,South,API,Referral,Medium,Approved,Level 4,2020-02-13,South,1,Gaming,Active
4,P000005,Prairie PayTech 0005,Merchant Acquirer,Healthcare,SMB,Active,US,CA,San Diego,West,API,Direct,High,Approved,Level 4,2022-03-09,West,1,Healthcare,Active


In [6]:
query_library = {
    "Q001": {
        "intent": "technology_az_growth_above_20",
        "example_question": "Show me technology partners in Arizona with transaction growth above 20%.",
        "keywords": ["technology", "arizona", "az", "growth", "20"],
        "sql": """
            SELECT
                pm.partner_id,
                pm.partner_name,
                pm.partner_type,
                pm.industry_vertical,
                pm.state,
                pm.partner_status,
                mt.transaction_month,
                mt.growth_pct
            FROM partner_master pm
            JOIN partner_metrics mt
                ON pm.partner_id = mt.partner_id
            WHERE pm.partner_type LIKE '%Technology%'
              AND pm.state = 'AZ'
              AND mt.growth_pct > 20
            ORDER BY mt.growth_pct DESC;
        """,
        "citations": [
            "partner_master.partner_id",
            "partner_master.partner_name",
            "partner_master.partner_type",
            "partner_master.industry_vertical",
            "partner_master.state",
            "partner_metrics.transaction_month",
            "partner_metrics.growth_pct"
        ]
    },

    "Q002": {
        "intent": "strategic_pricing_partners",
        "example_question": "Which partners are on the Strategic pricing tier?",
        "keywords": ["strategic", "pricing", "tier", "partners"],
        "sql": """
            SELECT
                pm.partner_id,
                pm.partner_name,
                pm.partner_status,
                pp.pricing_tier,
                pp.recommended_pricing_plan_id,
                pp.approval_status
            FROM partner_master pm
            JOIN partner_pricing pp
                ON pm.partner_id = pp.partner_id
            WHERE pp.pricing_tier = 'STRATEGIC'
            ORDER BY pm.partner_name;
        """,
        "citations": [
            "partner_master.partner_id",
            "partner_master.partner_name",
            "partner_pricing.pricing_tier",
            "partner_pricing.recommended_pricing_plan_id",
            "partner_pricing.approval_status"
        ]
    },

    "Q003": {
        "intent": "pricing_reviews_due_90_days",
        "example_question": "Which partners have pricing reviews due in the next 90 days?",
        "keywords": ["pricing", "review", "due", "90", "days"],
        "sql": """
            SELECT
                pm.partner_id,
                pm.partner_name,
                pp.pricing_tier,
                pp.approval_status,
                pp.review_due_date,
                pp.contract_status
            FROM partner_master pm
            JOIN partner_pricing pp
                ON pm.partner_id = pp.partner_id
            WHERE date(pp.review_due_date) BETWEEN date('2026-06-28') AND date('2026-09-26')
            ORDER BY date(pp.review_due_date);
        """,
        "citations": [
            "partner_master.partner_id",
            "partner_master.partner_name",
            "partner_pricing.pricing_tier",
            "partner_pricing.approval_status",
            "partner_pricing.review_due_date",
            "partner_pricing.contract_status"
        ]
    },

    "Q004": {
        "intent": "highest_average_transaction_volume_by_industry",
        "example_question": "Which industry has the highest average transaction volume?",
        "keywords": ["industry", "highest", "average", "transaction", "volume"],
        "sql": """
            SELECT
                pm.industry_vertical,
                ROUND(AVG(mt.transaction_volume), 2) AS avg_transaction_volume
            FROM partner_master pm
            JOIN partner_metrics mt
                ON pm.partner_id = mt.partner_id
            GROUP BY pm.industry_vertical
            ORDER BY avg_transaction_volume DESC;
        """,
        "citations": [
            "partner_master.industry_vertical",
            "partner_metrics.transaction_volume"
        ]
    },

    "Q005": {
        "intent": "top_5_partners_by_revenue",
        "example_question": "Show the top 5 partners by revenue.",
        "keywords": ["top", "5", "partners", "revenue"],
        "sql": """
            SELECT
                pm.partner_id,
                pm.partner_name,
                ROUND(SUM(mt.revenue), 2) AS total_revenue
            FROM partner_master pm
            JOIN partner_metrics mt
                ON pm.partner_id = mt.partner_id
            GROUP BY pm.partner_id, pm.partner_name
            ORDER BY total_revenue DESC
            LIMIT 5;
        """,
        "citations": [
            "partner_master.partner_id",
            "partner_master.partner_name",
            "partner_metrics.revenue"
        ]
    },

    "Q006": {
        "intent": "active_partners_with_pricing_exceptions",
        "example_question": "Which active partners have pricing exceptions?",
        "keywords": ["active", "partners", "pricing", "exceptions"],
        "sql": """
            SELECT
                pm.partner_id,
                pm.partner_name,
                pm.partner_status,
                pp.pricing_tier,
                pp.exception_flag,
                pp.exception_type,
                pp.negotiated_bps,
                pp.approval_status
            FROM partner_master pm
            JOIN partner_pricing pp
                ON pm.partner_id = pp.partner_id
            WHERE pm.partner_status = 'Active'
              AND pp.exception_flag = 1
            ORDER BY pm.partner_name;
        """,
        "citations": [
            "partner_master.partner_id",
            "partner_master.partner_name",
            "partner_master.partner_status",
            "partner_pricing.pricing_tier",
            "partner_pricing.exception_flag",
            "partner_pricing.exception_type",
            "partner_pricing.negotiated_bps",
            "partner_pricing.approval_status"
        ]
    },

    "Q007": {
        "intent": "missing_demographic_information",
        "example_question": "Which partners have missing city or state information?",
        "keywords": ["missing", "city", "state", "demographic"],
        "sql": """
            SELECT
                partner_id,
                partner_name,
                city,
                state,
                region
            FROM partner_master
            WHERE city IS NULL
               OR state IS NULL
               OR TRIM(city) = ''
               OR TRIM(state) = '';
        """,
        "citations": [
            "partner_master.partner_id",
            "partner_master.partner_name",
            "partner_master.city",
            "partner_master.state",
            "partner_master.region"
        ]
    },

    "Q008": {
        "intent": "average_growth_by_pricing_tier",
        "example_question": "Compare average transaction growth by pricing tier.",
        "keywords": ["average", "growth", "pricing", "tier", "compare"],
        "sql": """
            SELECT
                pp.pricing_tier,
                ROUND(AVG(mt.growth_pct), 2) AS avg_growth_pct,
                COUNT(*) AS metric_record_count
            FROM partner_pricing pp
            JOIN partner_metrics mt
                ON pp.partner_id = mt.partner_id
            WHERE mt.growth_pct IS NOT NULL
            GROUP BY pp.pricing_tier
            ORDER BY avg_growth_pct DESC;
        """,
        "citations": [
            "partner_pricing.pricing_tier",
            "partner_metrics.growth_pct"
        ]
    },

    "Q009": {
        "intent": "enterprise_partners_california",
        "example_question": "Which enterprise partners are in California?",
        "keywords": ["enterprise", "partners", "california", "ca"],
        "sql": """
            SELECT
                partner_id,
                partner_name,
                partner_size,
                state,
                city,
                partner_status
            FROM partner_master
            WHERE partner_size = 'Enterprise'
              AND state = 'CA'
            ORDER BY partner_name;
        """,
        "citations": [
            "partner_master.partner_id",
            "partner_master.partner_name",
            "partner_master.partner_size",
            "partner_master.state",
            "partner_master.city",
            "partner_master.partner_status"
        ]
    },

    "Q010": {
        "intent": "declining_growth_latest_month",
        "example_question": "Which partners had declining transaction growth in the latest month?",
        "keywords": ["declining", "growth", "latest", "month"],
        "sql": """
            WITH latest_month AS (
                SELECT MAX(date(transaction_month)) AS max_month
                FROM partner_metrics
            )
            SELECT
                pm.partner_id,
                pm.partner_name,
                mt.transaction_month,
                mt.growth_pct
            FROM partner_master pm
            JOIN partner_metrics mt
                ON pm.partner_id = mt.partner_id
            WHERE date(mt.transaction_month) = (SELECT max_month FROM latest_month)
              AND mt.growth_pct < 0
            ORDER BY mt.growth_pct ASC;
        """,
        "citations": [
            "partner_master.partner_id",
            "partner_master.partner_name",
            "partner_metrics.transaction_month",
            "partner_metrics.growth_pct"
        ]
    }
}

In [7]:
query_library_rows = []

for qid, item in query_library.items():
    query_library_rows.append({
        "question_id": qid,
        "intent": item["intent"],
        "example_question": item["example_question"],
        "keywords": ", ".join(item["keywords"]),
        "sql": item["sql"],
        "citations": ", ".join(item["citations"])
    })

query_library_df = pd.DataFrame(query_library_rows)

query_library_df.to_csv(
    os.path.join(phase5_path, "phase5_query_library.csv"),
    index=False
)

query_library_df

,question_id,intent,example_question,keywords,sql,citations
0,Q001,technology_az_growth_above_20,Show me technology partners in Arizona with tr...,"technology, arizona, az, growth, 20",\n SELECT\n pm.partn...,"partner_master.partner_id, partner_master.part..."
1,Q002,strategic_pricing_partners,Which partners are on the Strategic pricing tier?,"strategic, pricing, tier, partners",\n SELECT\n pm.partn...,"partner_master.partner_id, partner_master.part..."
2,Q003,pricing_reviews_due_90_days,Which partners have pricing reviews due in the...,"pricing, review, due, 90, days",\n SELECT\n pm.partn...,"partner_master.partner_id, partner_master.part..."
3,Q004,highest_average_transaction_volume_by_industry,Which industry has the highest average transac...,"industry, highest, average, transaction, volume",\n SELECT\n pm.indus...,"partner_master.industry_vertical, partner_metr..."
4,Q005,top_5_partners_by_revenue,Show the top 5 partners by revenue.,"top, 5, partners, revenue",\n SELECT\n pm.partn...,"partner_master.partner_id, partner_master.part..."
5,Q006,active_partners_with_pricing_exceptions,Which active partners have pricing exceptions?,"active, partners, pricing, exceptions",\n SELECT\n pm.partn...,"partner_master.partner_id, partner_master.part..."
6,Q007,missing_demographic_information,Which partners have missing city or state info...,"missing, city, state, demographic",\n SELECT\n partner_...,"partner_master.partner_id, partner_master.part..."
7,Q008,average_growth_by_pricing_tier,Compare average transaction growth by pricing ...,"average, growth, pricing, tier, compare",\n SELECT\n pp.prici...,"partner_pricing.pricing_tier, partner_metrics...."
8,Q009,enterprise_partners_california,Which enterprise partners are in California?,"enterprise, partners, california, ca",\n SELECT\n partner_...,"partner_master.partner_id, partner_master.part..."
9,Q010,declining_growth_latest_month,Which partners had declining transaction growt...,"declining, growth, latest, month",\n WITH latest_month AS (\n ...,"partner_master.partner_id, partner_master.part..."


In [8]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def match_question_to_intent(user_question):
    """
    Matches a natural-language question to the best query template.
    Uses simple keyword scoring.
    """
    normalized_question = normalize_text(user_question)

    best_match = None
    best_score = 0

    for qid, item in query_library.items():
        score = 0

        for keyword in item["keywords"]:
            keyword_normalized = normalize_text(keyword)

            if keyword_normalized in normalized_question:
                score += 1

        if score > best_score:
            best_score = score
            best_match = qid

    if best_match is None or best_score == 0:
        return None, 0

    return best_match, best_score

In [9]:
test_question = "Show me technology partners in AZ with growth greater than 20 percent."

matched_qid, score = match_question_to_intent(test_question)

print("Matched Question ID:", matched_qid)
print("Score:", score)
print("Matched Intent:", query_library[matched_qid]["intent"])

Matched Question ID: Q001
Score: 4
Matched Intent: technology_az_growth_above_20


In [10]:
def validate_sql_safety(sql_query):
    """
    Ensures that only safe read-only SQL queries are allowed.
    Blocks INSERT, UPDATE, DELETE, DROP, ALTER, and other risky commands.
    """
    blocked_keywords = [
        "insert", "update", "delete", "drop", "alter",
        "truncate", "create", "replace", "attach", "detach"
    ]

    normalized_sql = normalize_text(sql_query)

    for keyword in blocked_keywords:
        if keyword in normalized_sql:
            return False, f"Blocked keyword found: {keyword}"

    return True, "SQL passed safety validation"

In [11]:
safe, message = validate_sql_safety(query_library["Q001"]["sql"])

print(safe)
print(message)

True
SQL passed safety validation


In [12]:
def generate_answer_summary(question_id, result_df):
    """
    Creates a simple business-friendly answer based on the matched question.
    """

    row_count = len(result_df)

    if row_count == 0:
        return "No matching records were found for this question based on the available data."

    if question_id == "Q001":
        return f"I found {row_count} Technology partner metric records in Arizona with transaction growth above 20%."

    elif question_id == "Q002":
        return f"I found {row_count} partners currently assigned to the Strategic pricing tier."

    elif question_id == "Q003":
        return f"I found {row_count} partners with pricing reviews due in the next 90 days."

    elif question_id == "Q004":
        top_industry = result_df.iloc[0]["industry_vertical"]
        top_volume = result_df.iloc[0]["avg_transaction_volume"]
        return f"The industry with the highest average transaction volume is {top_industry}, with an average transaction volume of ${top_volume:,.2f}."

    elif question_id == "Q005":
        top_partner = result_df.iloc[0]["partner_name"]
        top_revenue = result_df.iloc[0]["total_revenue"]
        return f"The top revenue-generating partner is {top_partner}, with total revenue of ${top_revenue:,.2f}."

    elif question_id == "Q006":
        return f"I found {row_count} active partners with pricing exceptions."

    elif question_id == "Q007":
        return f"I found {row_count} partners with missing city or state information."

    elif question_id == "Q008":
        top_tier = result_df.iloc[0]["pricing_tier"]
        top_growth = result_df.iloc[0]["avg_growth_pct"]
        return f"The pricing tier with the highest average transaction growth is {top_tier}, with average growth of {top_growth}%."

    elif question_id == "Q009":
        return f"I found {row_count} enterprise partners located in California."

    elif question_id == "Q010":
        return f"I found {row_count} partners with declining transaction growth in the latest month."

    else:
        return f"The query returned {row_count} records."

In [13]:
assistant_logs = []

def partnerlens_assistant(user_question, show_sql=True, show_citations=True, preview_rows=10):
    """
    PartnerLens Copilot prototype.
    Input: natural-language question.
    Output: answer summary, SQL query, results, and citations.
    """

    matched_qid, score = match_question_to_intent(user_question)

    if matched_qid is None:
        response = {
            "user_question": user_question,
            "matched_question_id": None,
            "match_score": score,
            "answer": "I could not match this question to the current PartnerLens query library.",
            "sql": None,
            "citations": [],
            "result": pd.DataFrame()
        }

        assistant_logs.append({
            "timestamp": datetime.now(),
            "user_question": user_question,
            "matched_question_id": None,
            "match_score": score,
            "row_count": 0,
            "status": "No match"
        })

        print(response["answer"])
        return response

    sql_query = query_library[matched_qid]["sql"]
    citations = query_library[matched_qid]["citations"]

    is_safe, safety_message = validate_sql_safety(sql_query)

    if not is_safe:
        response = {
            "user_question": user_question,
            "matched_question_id": matched_qid,
            "match_score": score,
            "answer": f"SQL safety validation failed: {safety_message}",
            "sql": sql_query,
            "citations": citations,
            "result": pd.DataFrame()
        }

        assistant_logs.append({
            "timestamp": datetime.now(),
            "user_question": user_question,
            "matched_question_id": matched_qid,
            "match_score": score,
            "row_count": 0,
            "status": "SQL blocked"
        })

        print(response["answer"])
        return response

    result_df = run_sql(sql_query)
    answer = generate_answer_summary(matched_qid, result_df)

    response = {
        "user_question": user_question,
        "matched_question_id": matched_qid,
        "match_score": score,
        "answer": answer,
        "sql": sql_query,
        "citations": citations,
        "result": result_df
    }

    assistant_logs.append({
        "timestamp": datetime.now(),
        "user_question": user_question,
        "matched_question_id": matched_qid,
        "match_score": score,
        "row_count": len(result_df),
        "status": "Success"
    })

    print("User Question:")
    print(user_question)

    print("\nMatched Intent:")
    print(query_library[matched_qid]["intent"])

    print("\nAnswer:")
    print(answer)

    if show_sql:
        print("\nSQL Used:")
        print(sql_query)

    if show_citations:
        print("\nSources Used:")
        for citation in citations:
            print("-", citation)

    print("\nResult Preview:")
    display(result_df.head(preview_rows))

    return response

In [14]:
response_1 = partnerlens_assistant(
    "Show me technology partners in Arizona with transaction growth above 20%."
)

User Question:
Show me technology partners in Arizona with transaction growth above 20%.

Matched Intent:
technology_az_growth_above_20

Answer:
I found 85 Technology partner metric records in Arizona with transaction growth above 20%.

SQL Used:

            SELECT
                pm.partner_id,
                pm.partner_name,
                pm.partner_type,
                pm.industry_vertical,
                pm.state,
                pm.partner_status,
                mt.transaction_month,
                mt.growth_pct
            FROM partner_master pm
            JOIN partner_metrics mt
                ON pm.partner_id = mt.partner_id
            WHERE pm.partner_type LIKE '%Technology%'
              AND pm.state = 'AZ'
              AND mt.growth_pct > 20
            ORDER BY mt.growth_pct DESC;
        

Sources Used:
- partner_master.partner_id
- partner_master.partner_name
- partner_master.partner_type
- partner_master.industry_vertical
- partner_master.state
- partner_met

,partner_id,partner_name,partner_type,industry_vertical,state,partner_status,transaction_month,growth_pct
0,P004892,Summit Connect 4892,Technology Platform,Travel,AZ,Active,2024-12-01,55.72
1,P003960,Clearwater Processing 3960,Technology Platform,Hospitality,AZ,Pilot,2024-12-01,43.32
2,P004952,Granite Processing 4952,Technology Platform,Travel,AZ,Active,2024-11-01,41.59
3,P004612,Sunrise Connect 4612,Technology Platform,Travel,AZ,Active,2025-11-01,41.22
4,P004952,Granite Processing 4952,Technology Platform,Travel,AZ,Active,2025-12-01,40.97
5,P002009,Harbor Solutions 2009,Technology Platform,SaaS,AZ,Active,2024-02-01,37.18
6,P002733,Sunrise Market 2733,Technology Platform,Retail,AZ,Suspended,2025-10-01,36.61
7,P001717,Oakstone Gateway 1717,Technology Platform,Travel,AZ,Active,2024-11-01,36.24
8,P001160,Granite Processing 1160,Technology Platform,Hospitality,AZ,Active,2024-12-01,35.97
9,P003711,Granite Gateway 3711,Technology Platform,Travel,AZ,Active,2025-11-01,34.95


In [15]:
response_2 = partnerlens_assistant(
    "Which partners are on the Strategic pricing tier?"
)

User Question:
Which partners are on the Strategic pricing tier?

Matched Intent:
strategic_pricing_partners

Answer:
I found 391 partners currently assigned to the Strategic pricing tier.

SQL Used:

            SELECT
                pm.partner_id,
                pm.partner_name,
                pm.partner_status,
                pp.pricing_tier,
                pp.recommended_pricing_plan_id,
                pp.approval_status
            FROM partner_master pm
            JOIN partner_pricing pp
                ON pm.partner_id = pp.partner_id
            WHERE pp.pricing_tier = 'STRATEGIC'
            ORDER BY pm.partner_name;
        

Sources Used:
- partner_master.partner_id
- partner_master.partner_name
- partner_pricing.pricing_tier
- partner_pricing.recommended_pricing_plan_id
- partner_pricing.approval_status

Result Preview:


,partner_id,partner_name,partner_status,pricing_tier,recommended_pricing_plan_id,approval_status
0,P000760,BluePeak Billing 0760,Active,STRATEGIC,STRATEGIC,Not Required
1,P002523,BluePeak Billing 2523,Active,STRATEGIC,STRATEGIC,Not Required
2,P004400,BluePeak Billing 4400,Active,STRATEGIC,STRATEGIC,Not Required
3,P004791,BluePeak Billing 4791,Active,STRATEGIC,STRATEGIC,Not Required
4,P001740,BluePeak Checkout 1740,Active,STRATEGIC,STRATEGIC,Not Required
5,P004440,BluePeak Commerce 4440,Active,STRATEGIC,STRATEGIC,Not Required
6,P004673,BluePeak Commerce 4673,Active,STRATEGIC,GROWTH,Approved
7,P002564,BluePeak Financial 2564,Active,STRATEGIC,STRATEGIC,Not Required
8,P000116,BluePeak Gateway 0116,Active,STRATEGIC,STRATEGIC,Not Required
9,P000462,BluePeak Gateway 0462,Active,STRATEGIC,STRATEGIC,Not Required


In [16]:
response_3 = partnerlens_assistant(
    "Which partners have pricing reviews due in the next 90 days?"
)

User Question:
Which partners have pricing reviews due in the next 90 days?

Matched Intent:
pricing_reviews_due_90_days

Answer:
I found 361 partners with pricing reviews due in the next 90 days.

SQL Used:

            SELECT
                pm.partner_id,
                pm.partner_name,
                pp.pricing_tier,
                pp.approval_status,
                pp.review_due_date,
                pp.contract_status
            FROM partner_master pm
            JOIN partner_pricing pp
                ON pm.partner_id = pp.partner_id
            WHERE date(pp.review_due_date) BETWEEN date('2026-06-28') AND date('2026-09-26')
            ORDER BY date(pp.review_due_date);
        

Sources Used:
- partner_master.partner_id
- partner_master.partner_name
- partner_pricing.pricing_tier
- partner_pricing.approval_status
- partner_pricing.review_due_date
- partner_pricing.contract_status

Result Preview:


,partner_id,partner_name,pricing_tier,approval_status,review_due_date,contract_status
0,P000170,Keystone PayWorks 0170,LAUNCH,Not Required,2026-06-28,Review Due Soon
1,P000368,Skyline Market 0368,PLATFORM_ISV,Not Required,2026-06-28,Review Due Soon
2,P003504,Cedar PayWorks 3504,GROWTH,Not Required,2026-06-28,Review Due Soon
3,P003984,Evergreen Financial 3984,LAUNCH,Not Required,2026-06-28,Review Due Soon
4,P004509,Prairie Payments 4509,PROMO_GROWTH,Approved,2026-06-28,Review Due Soon
5,P004928,Cedar Solutions 4928,LAUNCH,Not Required,2026-06-28,Review Due Soon
6,P000980,Clearwater Merchant Services 0980,SCALE,Approved,2026-06-29,Review Due Soon
7,P001302,Summit Market 1302,GROWTH,Not Required,2026-06-29,Review Due Soon
8,P001751,Cedar Market 1751,PLATFORM_ISV,Not Required,2026-06-29,Review Due Soon
9,P002144,BluePeak Connect 2144,HIGH_RISK,Not Required,2026-06-29,Review Due Soon


In [17]:
response_4 = partnerlens_assistant(
    "Which industry has the highest average transaction volume?"
)

User Question:
Which industry has the highest average transaction volume?

Matched Intent:
highest_average_transaction_volume_by_industry

Answer:
The industry with the highest average transaction volume is Gaming, with an average transaction volume of $5,018,721.46.

SQL Used:

            SELECT
                pm.industry_vertical,
                ROUND(AVG(mt.transaction_volume), 2) AS avg_transaction_volume
            FROM partner_master pm
            JOIN partner_metrics mt
                ON pm.partner_id = mt.partner_id
            GROUP BY pm.industry_vertical
            ORDER BY avg_transaction_volume DESC;
        

Sources Used:
- partner_master.industry_vertical
- partner_metrics.transaction_volume

Result Preview:


,industry_vertical,avg_transaction_volume
0,Gaming,5018721.46
1,Travel,4521112.13
2,Fintech,4449626.97
3,Utilities,4153576.08
4,Logistics,4149628.57
5,Healthcare,3949840.39
6,Government,3896428.26
7,Hospitality,3887502.99
8,Retail,3876466.52
9,Professional Services,3814276.01


In [18]:
response_5 = partnerlens_assistant(
    "Show the top 5 partners by revenue."
)

User Question:
Show the top 5 partners by revenue.

Matched Intent:
top_5_partners_by_revenue

Answer:
The top revenue-generating partner is Skyline Payments 0610, with total revenue of $64,847,292.41.

SQL Used:

            SELECT
                pm.partner_id,
                pm.partner_name,
                ROUND(SUM(mt.revenue), 2) AS total_revenue
            FROM partner_master pm
            JOIN partner_metrics mt
                ON pm.partner_id = mt.partner_id
            GROUP BY pm.partner_id, pm.partner_name
            ORDER BY total_revenue DESC
            LIMIT 5;
        

Sources Used:
- partner_master.partner_id
- partner_master.partner_name
- partner_metrics.revenue

Result Preview:


,partner_id,partner_name,total_revenue
0,P000610,Skyline Payments 0610,64847292.41
1,P002651,RedRock PayTech 2651,28539531.11
2,P003122,Cedar Market 3122,27485493.95
3,P004602,NorthStar Commerce 4602,26392278.10
4,P001422,Brightpath Platform 1422,25564051.57


In [19]:
response_6 = partnerlens_assistant(
    "Which active partners have pricing exceptions?"
)

User Question:
Which active partners have pricing exceptions?

Matched Intent:
active_partners_with_pricing_exceptions

Answer:
I found 681 active partners with pricing exceptions.

SQL Used:

            SELECT
                pm.partner_id,
                pm.partner_name,
                pm.partner_status,
                pp.pricing_tier,
                pp.exception_flag,
                pp.exception_type,
                pp.negotiated_bps,
                pp.approval_status
            FROM partner_master pm
            JOIN partner_pricing pp
                ON pm.partner_id = pp.partner_id
            WHERE pm.partner_status = 'Active'
              AND pp.exception_flag = 1
            ORDER BY pm.partner_name;
        

Sources Used:
- partner_master.partner_id
- partner_master.partner_name
- partner_master.partner_status
- partner_pricing.pricing_tier
- partner_pricing.exception_flag
- partner_pricing.exception_type
- partner_pricing.negotiated_bps
- partner_pricing.approval_

,partner_id,partner_name,partner_status,pricing_tier,exception_flag,exception_type,negotiated_bps,approval_status
0,P000573,BluePeak Billing 0573,Active,LEGACY_STANDARD,1,Legacy plan extension,53.65,Approved
1,P000735,BluePeak Billing 0735,Active,PLATFORM_ISV,1,High-risk waiver,55.48,Approved
2,P004415,BluePeak Billing 4415,Active,GROWTH,1,Volume threshold waiver,52.99,Pending
3,P002322,BluePeak Checkout 2322,Active,LEGACY_STANDARD,1,Waived monthly minimum,67.65,Pending
4,P002996,BluePeak Checkout 2996,Active,SCALE,1,High-risk waiver,48.18,Approved
5,P004673,BluePeak Commerce 4673,Active,STRATEGIC,1,Legacy plan extension,15.39,Approved
6,P001468,BluePeak Connect 1468,Active,HIGH_RISK,1,High-risk waiver,113.58,Approved
7,P004552,BluePeak Connect 4552,Active,LAUNCH,1,Legacy plan extension,61.46,Approved
8,P004675,BluePeak Connect 4675,Active,LEGACY_STANDARD,1,High-risk waiver,73.08,Pending
9,P000862,BluePeak Financial 0862,Active,GROWTH,1,Waived monthly minimum,70.61,Approved


In [20]:
response_7 = partnerlens_assistant(
    "Which partners have missing city or state information?"
)

User Question:
Which partners have missing city or state information?

Matched Intent:
missing_demographic_information

Answer:
No matching records were found for this question based on the available data.

SQL Used:

            SELECT
                partner_id,
                partner_name,
                city,
                state,
                region
            FROM partner_master
            WHERE city IS NULL
               OR state IS NULL
               OR TRIM(city) = ''
               OR TRIM(state) = '';
        

Sources Used:
- partner_master.partner_id
- partner_master.partner_name
- partner_master.city
- partner_master.state
- partner_master.region

Result Preview:


,partner_id,partner_name,city,state,region


In [21]:
response_8 = partnerlens_assistant(
    "Compare average transaction growth by pricing tier."
)

User Question:
Compare average transaction growth by pricing tier.

Matched Intent:
average_growth_by_pricing_tier

Answer:
The pricing tier with the highest average transaction growth is PLATFORM_ISV, with average growth of 2.97%.

SQL Used:

            SELECT
                pp.pricing_tier,
                ROUND(AVG(mt.growth_pct), 2) AS avg_growth_pct,
                COUNT(*) AS metric_record_count
            FROM partner_pricing pp
            JOIN partner_metrics mt
                ON pp.partner_id = mt.partner_id
            WHERE mt.growth_pct IS NOT NULL
            GROUP BY pp.pricing_tier
            ORDER BY avg_growth_pct DESC;
        

Sources Used:
- partner_pricing.pricing_tier
- partner_metrics.growth_pct

Result Preview:


,pricing_tier,avg_growth_pct,metric_record_count
0,PLATFORM_ISV,2.97,16399
1,LEGACY_STANDARD,2.77,2967
2,STRATEGIC,2.44,8993
3,HIGH_RISK,2.42,19803
4,SCALE,2.40,13662
5,GROWTH,2.09,24219
6,PROMO_GROWTH,2.00,943
7,LEGACY_ENTERPRISE,2.00,3197
8,LAUNCH,1.36,24817


In [22]:
response_9 = partnerlens_assistant(
    "Which enterprise partners are in California?"
)

User Question:
Which enterprise partners are in California?

Matched Intent:
enterprise_partners_california

Answer:
I found 145 enterprise partners located in California.

SQL Used:

            SELECT
                partner_id,
                partner_name,
                partner_size,
                state,
                city,
                partner_status
            FROM partner_master
            WHERE partner_size = 'Enterprise'
              AND state = 'CA'
            ORDER BY partner_name;
        

Sources Used:
- partner_master.partner_id
- partner_master.partner_name
- partner_master.partner_size
- partner_master.state
- partner_master.city
- partner_master.partner_status

Result Preview:


,partner_id,partner_name,partner_size,state,city,partner_status
0,P001362,BluePeak Checkout 1362,Enterprise,CA,Sacramento,Active
1,P001393,BluePeak Commerce 1393,Enterprise,CA,San Diego,Dormant
2,P003724,BluePeak Connect 3724,Enterprise,CA,San Diego,Active
3,P002564,BluePeak Financial 2564,Enterprise,CA,San Jose,Active
4,P001412,BluePeak Merchant Services 1412,Enterprise,CA,San Diego,Dormant
5,P000603,BluePeak Solutions 0603,Enterprise,CA,Los Angeles,Active
6,P000867,Brightpath Billing 0867,Enterprise,CA,Los Angeles,Pilot
7,P003930,Brightpath Billing 3930,Enterprise,CA,San Diego,Active
8,P004429,Brightpath Connect 4429,Enterprise,CA,Sacramento,Active
9,P004379,Brightpath Merchant Services 4379,Enterprise,CA,San Diego,Dormant


In [23]:
response_10 = partnerlens_assistant(
    "Which partners had declining transaction growth in the latest month?"
)

User Question:
Which partners had declining transaction growth in the latest month?

Matched Intent:
declining_growth_latest_month

Answer:
I found 1298 partners with declining transaction growth in the latest month.

SQL Used:

            WITH latest_month AS (
                SELECT MAX(date(transaction_month)) AS max_month
                FROM partner_metrics
            )
            SELECT
                pm.partner_id,
                pm.partner_name,
                mt.transaction_month,
                mt.growth_pct
            FROM partner_master pm
            JOIN partner_metrics mt
                ON pm.partner_id = mt.partner_id
            WHERE date(mt.transaction_month) = (SELECT max_month FROM latest_month)
              AND mt.growth_pct < 0
            ORDER BY mt.growth_pct ASC;
        

Sources Used:
- partner_master.partner_id
- partner_master.partner_name
- partner_metrics.transaction_month
- partner_metrics.growth_pct

Result Preview:


,partner_id,partner_name,transaction_month,growth_pct
0,P002649,Sunrise Billing 2649,2025-12-01,-27.27
1,P000524,BluePeak Processing 0524,2025-12-01,-26.10
2,P004263,Oakstone Payments 4263,2025-12-01,-25.77
3,P002921,RedRock Payments 2921,2025-12-01,-25.10
4,P002253,Maple Checkout 2253,2025-12-01,-24.96
5,P000946,Skyline Merchant Services 0946,2025-12-01,-24.17
6,P000789,Cedar PayTech 0789,2025-12-01,-23.76
7,P004693,Brightpath Market 4693,2025-12-01,-23.40
8,P000394,Granite PayWorks 0394,2025-12-01,-23.07
9,P004157,Summit Processing 4157,2025-12-01,-22.79


In [24]:
test_questions = [
    "Find technology partners in AZ where growth is more than 20 percent.",
    "List all partners with strategic pricing.",
    "Show pricing reviews coming due soon.",
    "What industry has the largest average payment volume?",
    "Who are the top five partners by revenue?",
    "Find active partners that have pricing exceptions.",
    "Show me partners missing city or state.",
    "Compare growth across pricing tiers.",
    "List enterprise partners in CA.",
    "Show partners with negative growth in the most recent month."
]

for question in test_questions:
    print("=" * 100)
    partnerlens_assistant(question, show_sql=False, show_citations=True, preview_rows=5)

User Question:
Find technology partners in AZ where growth is more than 20 percent.

Matched Intent:
technology_az_growth_above_20

Answer:
I found 85 Technology partner metric records in Arizona with transaction growth above 20%.

Sources Used:
- partner_master.partner_id
- partner_master.partner_name
- partner_master.partner_type
- partner_master.industry_vertical
- partner_master.state
- partner_metrics.transaction_month
- partner_metrics.growth_pct

Result Preview:


,partner_id,partner_name,partner_type,industry_vertical,state,partner_status,transaction_month,growth_pct
0,P004892,Summit Connect 4892,Technology Platform,Travel,AZ,Active,2024-12-01,55.72
1,P003960,Clearwater Processing 3960,Technology Platform,Hospitality,AZ,Pilot,2024-12-01,43.32
2,P004952,Granite Processing 4952,Technology Platform,Travel,AZ,Active,2024-11-01,41.59
3,P004612,Sunrise Connect 4612,Technology Platform,Travel,AZ,Active,2025-11-01,41.22
4,P004952,Granite Processing 4952,Technology Platform,Travel,AZ,Active,2025-12-01,40.97


User Question:
List all partners with strategic pricing.

Matched Intent:
strategic_pricing_partners

Answer:
I found 391 partners currently assigned to the Strategic pricing tier.

Sources Used:
- partner_master.partner_id
- partner_master.partner_name
- partner_pricing.pricing_tier
- partner_pricing.recommended_pricing_plan_id
- partner_pricing.approval_status

Result Preview:


,partner_id,partner_name,partner_status,pricing_tier,recommended_pricing_plan_id,approval_status
0,P000760,BluePeak Billing 0760,Active,STRATEGIC,STRATEGIC,Not Required
1,P002523,BluePeak Billing 2523,Active,STRATEGIC,STRATEGIC,Not Required
2,P004400,BluePeak Billing 4400,Active,STRATEGIC,STRATEGIC,Not Required
3,P004791,BluePeak Billing 4791,Active,STRATEGIC,STRATEGIC,Not Required
4,P001740,BluePeak Checkout 1740,Active,STRATEGIC,STRATEGIC,Not Required


User Question:
Show pricing reviews coming due soon.

Matched Intent:
pricing_reviews_due_90_days

Answer:
I found 361 partners with pricing reviews due in the next 90 days.

Sources Used:
- partner_master.partner_id
- partner_master.partner_name
- partner_pricing.pricing_tier
- partner_pricing.approval_status
- partner_pricing.review_due_date
- partner_pricing.contract_status

Result Preview:


...

,pricing_tier,avg_growth_pct,metric_record_count
0,PLATFORM_ISV,2.97,16399
1,LEGACY_STANDARD,2.77,2967
2,STRATEGIC,2.44,8993
3,HIGH_RISK,2.42,19803
4,SCALE,2.40,13662


User Question:
List enterprise partners in CA.

Matched Intent:
enterprise_partners_california

Answer:
I found 145 enterprise partners located in California.

Sources Used:
- partner_master.partner_id
- partner_master.partner_name
- partner_master.partner_size
- partner_master.state
- partner_master.city
- partner_master.partner_status

Result Preview:


,partner_id,partner_name,partner_size,state,city,partner_status
0,P001362,BluePeak Checkout 1362,Enterprise,CA,Sacramento,Active
1,P001393,BluePeak Commerce 1393,Enterprise,CA,San Diego,Dormant
2,P003724,BluePeak Connect 3724,Enterprise,CA,San Diego,Active
3,P002564,BluePeak Financial 2564,Enterprise,CA,San Jose,Active
4,P001412,BluePeak Merchant Services 1412,Enterprise,CA,San Diego,Dormant


User Question:
Show partners with negative growth in the most recent month.

Matched Intent:
declining_growth_latest_month

Answer:
I found 1298 partners with declining transaction growth in the latest month.

Sources Used:
- partner_master.partner_id
- partner_master.partner_name
- partner_metrics.transaction_month
- partner_metrics.growth_pct

Result Preview:


,partner_id,partner_name,transaction_month,growth_pct
0,P002649,Sunrise Billing 2649,2025-12-01,-27.27
1,P000524,BluePeak Processing 0524,2025-12-01,-26.10
2,P004263,Oakstone Payments 4263,2025-12-01,-25.77
3,P002921,RedRock Payments 2921,2025-12-01,-25.10
4,P002253,Maple Checkout 2253,2025-12-01,-24.96


In [25]:
assistant_logs_df = pd.DataFrame(assistant_logs)

assistant_logs_df.to_csv(
    os.path.join(phase5_path, "phase5_assistant_logs.csv"),
    index=False
)

assistant_logs_df

,timestamp,user_question,matched_question_id,match_score,row_count,status
0,2026-07-05 00:37:01.801568,Show me technology partners in Arizona with tr...,Q001,4,85,Success
1,2026-07-05 00:37:08.218386,Which partners are on the Strategic pricing tier?,Q002,4,391,Success
2,2026-07-05 00:37:21.546855,Which partners have pricing reviews due in the...,Q003,5,361,Success
3,2026-07-05 00:37:29.861992,Which industry has the highest average transac...,Q004,5,13,Success
4,2026-07-05 00:37:37.533848,Show the top 5 partners by revenue.,Q005,4,5,Success
5,2026-07-05 00:37:47.998308,Which active partners have pricing exceptions?,Q006,4,681,Success
6,2026-07-05 00:37:55.126452,Which partners have missing city or state info...,Q007,3,0,Success
7,2026-07-05 00:38:12.678260,Compare average transaction growth by pricing ...,Q008,5,9,Success
8,2026-07-05 00:38:19.547539,Which enterprise partners are in California?,Q009,4,145,Success
9,2026-07-05 00:38:26.379126,Which partners had declining transaction growt...,Q010,4,1298,Success


In [26]:
phase5_test_results = []

for question in test_questions:
    matched_qid, score = match_question_to_intent(question)

    if matched_qid is not None:
        sql_query = query_library[matched_qid]["sql"]
        result_df = run_sql(sql_query)
        row_count = len(result_df)
        status = "Matched"
        intent = query_library[matched_qid]["intent"]
    else:
        row_count = 0
        status = "No match"
        intent = None

    phase5_test_results.append({
        "user_question": question,
        "matched_question_id": matched_qid,
        "matched_intent": intent,
        "match_score": score,
        "result_row_count": row_count,
        "status": status
    })

phase5_test_results_df = pd.DataFrame(phase5_test_results)

phase5_test_results_df.to_csv(
    os.path.join(phase5_path, "phase5_question_test_results.csv"),
    index=False
)

phase5_test_results_df

,user_question,matched_question_id,matched_intent,match_score,result_row_count,status
0,Find technology partners in AZ where growth is...,Q001,technology_az_growth_above_20,4,85,Matched
1,List all partners with strategic pricing.,Q002,strategic_pricing_partners,3,391,Matched
2,Show pricing reviews coming due soon.,Q003,pricing_reviews_due_90_days,3,361,Matched
3,What industry has the largest average payment ...,Q004,highest_average_transaction_volume_by_industry,3,13,Matched
4,Who are the top five partners by revenue?,Q005,top_5_partners_by_revenue,3,5,Matched
5,Find active partners that have pricing excepti...,Q006,active_partners_with_pricing_exceptions,4,681,Matched
6,Show me partners missing city or state.,Q007,missing_demographic_information,3,0,Matched
7,Compare growth across pricing tiers.,Q008,average_growth_by_pricing_tier,4,9,Matched
8,List enterprise partners in CA.,Q009,enterprise_partners_california,3,145,Matched
9,Show partners with negative growth in the most...,Q010,declining_growth_latest_month,2,1298,Matched


In [27]:
matched_count = (phase5_test_results_df["status"] == "Matched").sum()
total_questions = len(phase5_test_results_df)

match_rate = matched_count / total_questions

print("Matched questions:", matched_count)
print("Total questions:", total_questions)
print("Prototype match rate:", round(match_rate * 100, 2), "%")

Matched questions: 10
Total questions: 10
Prototype match rate: 100.0 %


In [28]:
summary_text = f"""
Phase 5 Summary: PartnerLens Assistant Prototype

Phase 5 focused on building a controlled natural-language-to-SQL assistant prototype for PartnerLens Copilot.

The assistant uses a query library of approved SQL templates mapped to common partner intelligence business questions. A keyword-based intent matcher identifies the most relevant business question, retrieves the matching SQL query, validates it for read-only safety, executes it against the PartnerLens SQLite database, and returns a business-friendly answer with source-field citations.

Key capabilities completed:
1. Natural-language question input
2. Intent matching
3. SQL template selection
4. SQL safety validation
5. SQLite query execution
6. Answer summary generation
7. Source-field citation display
8. Assistant interaction logging

Prototype test results:
- Total test questions: {total_questions}
- Matched questions: {matched_count}
- Match rate: {round(match_rate * 100, 2)}%

This controlled SQL-template approach was selected for the MVP because it is safer, more auditable, and easier to evaluate than unrestricted LLM-generated SQL. The next phase will focus on improving citation auditing, guardrails, and unsupported-question handling.
"""

summary_file = os.path.join(phase5_path, "phase5_summary.txt")

with open(summary_file, "w") as f:
    f.write(summary_text)

print(summary_text)
print("Summary saved to:", summary_file)


Phase 5 Summary: PartnerLens Assistant Prototype

Phase 5 focused on building a controlled natural-language-to-SQL assistant prototype for PartnerLens Copilot.

The assistant uses a query library of approved SQL templates mapped to common partner intelligence business questions. A keyword-based intent matcher identifies the most relevant business question, retrieves the matching SQL query, validates it for read-only safety, executes it against the PartnerLens SQLite database, and returns a business-friendly answer with source-field citations.

Key capabilities completed:
1. Natural-language question input
2. Intent matching
3. SQL template selection
4. SQL safety validation
5. SQLite query execution
6. Answer summary generation
7. Source-field citation display
8. Assistant interaction logging

Prototype test results:
- Total test questions: 10
- Matched questions: 10
- Match rate: 100.0%

This controlled SQL-template approach was selected for the MVP because it is safer, more auditabl

In [29]:
conn.close()
print("SQLite connection closed.")

SQLite connection closed.
